In [20]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg
import time

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Spark Architecture Demo") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

In [21]:
sales_data = [
    ("2023-01-01", "Electronics", "Laptop", 1200, "North"),
    ("2023-01-01", "Electronics", "Phone", 800, "South"),
    ("2023-01-02", "Clothing", "Shirt", 50, "North"),
    ("2023-01-02", "Electronics", "Tablet", 600, "East"),
    ("2023-01-03", "Clothing", "Pants", 80, "West"),
    ("2023-01-03", "Electronics", "Laptop", 1200, "South"),
    ("2023-01-04", "Food", "Coffee", 15, "North"),
    ("2023-01-04", "Food", "Sandwich", 12, "East"),
] * 1000

In [22]:
df = spark.createDataFrame(sales_data,
                          ["date", "category", "product", "amount", "region"])

In [23]:
df.printSchema()

root
 |-- date: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- region: string (nullable = true)



In [24]:
print("\n=== STAGE 1: REPARTITION ===")
print(f"Original partitions: {df.rdd.getNumPartitions()}")



=== STAGE 1: REPARTITION ===
Original partitions: 2


In [25]:
df_repartitioned = df.repartition(4, col("region"))
print(f"After repartitioning: {df_repartitioned.rdd.getNumPartitions()}")

After repartitioning: 4


In [26]:
print("\n=== STAGE 2: FILTERING AND GROUPING ===")
filtered_df = df_repartitioned.where(col("amount") > 100)
print("Applied WHERE filter for amount > 100")


=== STAGE 2: FILTERING AND GROUPING ===
Applied WHERE filter for amount > 100


In [27]:
selected_df = filtered_df.select("category", "region", "amount")
print("Applied SELECT for category, region, amount")

Applied SELECT for category, region, amount


In [28]:
grouped_df = selected_df.groupBy("category", "region").agg(
    count("*").alias("transaction_count"),
    spark_sum("amount").alias("total_amount"),
    avg("amount").alias("avg_amount")
)
print("Applied GROUP BY on category and region")

Applied GROUP BY on category and region


In [29]:
print("\n=== STAGE 3: FINAL PROCESSING ===")

final_results = grouped_df.withColumn("revenue_category",
    col("total_amount") / col("transaction_count"))


print("Applied final calculations")





=== STAGE 3: FINAL PROCESSING ===
Applied final calculations


In [ ]:
# ===== EXECUTION AND MONITORING =====
print("\n=== EXECUTING THE JOB ===")
print("Note: Up to this point, no actual computation has happened due to lazy evaluation!")


In [30]:
start_time = time.time()
results = final_results.collect()  # Action: This triggers execution!
end_time = time.time()

print(f"Job completed in {end_time - start_time:.2f} seconds")
print(f"Results count: {len(results)}")

Job completed in 1.51 seconds
Results count: 3


In [31]:
# ===== UNDERSTANDING TASK DISTRIBUTION =====
print("\n--- LOGICAL PLAN ---")
final_results.explain(mode="simple")



--- LOGICAL PLAN ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   *(2) Project [category#57, region#60, transaction_count#73L, total_amount#75L, avg_amount#77, (cast(total_amount#75L as double) / cast(transaction_count#73L as double)) AS revenue_category#83]
   +- *(2) HashAggregate(keys=[category#57, region#60], functions=[count(1), sum(amount#59L), avg(amount#59L)])
      +- *(2) HashAggregate(keys=[category#57, region#60], functions=[partial_count(1), partial_sum(amount#59L), partial_avg(amount#59L)])
         +- *(2) Project [category#57, region#60, amount#59L]
            +- ShuffleQueryStage 0
               +- Exchange hashpartitioning(region#60, 4), REPARTITION_BY_NUM, [plan_id=167]
                  +- *(1) Project [category#57, amount#59L, region#60]
                     +- *(1) Filter (isnotnull(amount#59L) AND (amount#59L > 100))
                        +- *(1) Scan ExistingRDD[date#56,category#57,product#58,amount#59L,region#60]
+- == Init

In [32]:

print("\n--- PHYSICAL PLAN (shows actual execution strategy) ---")
final_results.explain(mode="formatted")


--- PHYSICAL PLAN (shows actual execution strategy) ---
== Physical Plan ==
AdaptiveSparkPlan (15)
+- == Final Plan ==
   * Project (9)
   +- * HashAggregate (8)
      +- * HashAggregate (7)
         +- * Project (6)
            +- ShuffleQueryStage (5), Statistics(sizeInBytes=218.8 KiB, rowCount=4.00E+3)
               +- Exchange (4)
                  +- * Project (3)
                     +- * Filter (2)
                        +- * Scan ExistingRDD (1)
+- == Initial Plan ==
   Project (14)
   +- HashAggregate (13)
      +- HashAggregate (12)
         +- Project (11)
            +- Exchange (10)
               +- Project (3)
                  +- Filter (2)
                     +- Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [5]: [date#56, category#57, product#58, amount#59L, region#60]
Arguments: [date#56, category#57, product#58, amount#59L, region#60], MapPartitionsRDD[30] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPart